In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import gc
from pathlib import Path
import time
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
# ==============================================================================
# PASO 1: CONFIGURACIÓN DE RUTAS Y REGLAS DE HOMOLOGACIÓN
# ==============================================================================
FOLDER_PATH = Path(
    r"/content/drive/MyDrive/Maestria Ingenieria y analitica de datos/semestre 4/proyecto de grado/fuentes de datos/base_estudio"
)
COLUMN_HOMOLOGATION = {
    "estu_etnia": "estu_grupoetnia",
    "estu_tieneetnia": "estu_grupoetnia",
}

# Códigos DANE objetivo (11: Bogotá D.C., 25: Cundinamarca)
# Si necesitas 15 específicamente por tu diseño de datos, inclúyelo en la lista:
COD_DEPTOS_OBJETIVO = ["11", "25"]

txt_files = sorted(list(FOLDER_PATH.glob("Examen_Saber_11_*.txt")))
print(f"--- PASO 1: Se detectaron {len(txt_files)} archivos de microdatos (.txt) ---")

# ==============================================================================
# PASO 2: LECTURA, NORMALIZACIÓN Y FILTRADO REGIONAL INDIVIDUAL
# ==============================================================================
lista_dfs = []
start_time = time.time()

print("\n--- PASO 2: Procesamiento y filtrado por cohorte ---")

for file_path in txt_files:
    periodo_str = file_path.stem.replace("Examen_Saber_11_", "")

    # 2.1 Carga rápida como string para evitar inconsistencias de dtypes
    df_temp = pd.read_csv(
        file_path, sep=";", encoding="utf-8-sig", low_memory=False, dtype=str
    )

    # 2.2 Normalización sintáctica de columnas
    df_temp.columns = df_temp.columns.str.strip().str.lower()

    # 2.3 Desduplicación de columnas de origen (antes de renombrar)
    df_temp = df_temp.loc[:, ~df_temp.columns.duplicated(keep="first")]

    # 2.4 Homologación nomenclatural
    df_temp.rename(columns=COLUMN_HOMOLOGATION, inplace=True)

    # 2.5 Desduplicación post-renombrado
    df_temp = df_temp.loc[:, ~df_temp.columns.duplicated(keep="first")]

    # 2.6 FILTRO REGIONAL POR CÓDIGO DANE
    if "cole_cod_depto_ubicacion" in df_temp.columns:
        # Limpieza: eliminar espacios y sufijos decimales (.0) si existieran
        df_temp["cole_cod_depto_ubicacion"] = (
            df_temp["cole_cod_depto_ubicacion"]
            .astype(str)
            .str.strip()
            .str.split(".")
            .str[0]
            .str.zfill(2) # Estandariza a 2 dígitos ('11', '25', etc.)
        )
        df_temp = df_temp[df_temp["cole_cod_depto_ubicacion"].isin(COD_DEPTOS_OBJETIVO)]

    # 2.7 Inyección del periodo
    df_temp["periodo_ejecucion"] = int(periodo_str)

    # Reset de índice preventivo por cohorte
    df_temp.reset_index(drop=True, inplace=True)

    print(
        f" [✓] Cohorte {periodo_str}: {df_temp.shape[0]:,} registros filtrados | {df_temp.shape[1]} columnas"
    )

    lista_dfs.append(df_temp)

print(f"\n⏱️ Lectura y filtrado completados en {time.time() - start_time:.2f} segundos.")
print(f"Lista `lista_dfs` lista en memoria con las {len(lista_dfs)} cohortes filtradas.")

--- PASO 1: Se detectaron 14 archivos de microdatos (.txt) ---

--- PASO 2: Procesamiento y filtrado por cohorte ---
 [✓] Cohorte 20191: 6,568 registros filtrados | 85 columnas
 [✓] Cohorte 20192: 122,063 registros filtrados | 85 columnas
 [✓] Cohorte 20201: 4,671 registros filtrados | 85 columnas
 [✓] Cohorte 20202: 113,500 registros filtrados | 85 columnas
 [✓] Cohorte 20211: 4,403 registros filtrados | 84 columnas
 [✓] Cohorte 20212: 115,892 registros filtrados | 85 columnas
 [✓] Cohorte 20221: 6,259 registros filtrados | 84 columnas
 [✓] Cohorte 20222: 114,000 registros filtrados | 84 columnas
 [✓] Cohorte 20231: 6,313 registros filtrados | 84 columnas
 [✓] Cohorte 20232: 117,978 registros filtrados | 84 columnas
 [✓] Cohorte 20241: 6,083 registros filtrados | 84 columnas
 [✓] Cohorte 20242: 118,481 registros filtrados | 84 columnas
 [✓] Cohorte 20251: 6,641 registros filtrados | 84 columnas
 [✓] Cohorte 20252: 118,399 registros filtrados | 91 columnas

⏱️ Lectura y filtrado comple

In [ ]:
"""
==============================================================================
PASO 3 Y 4: CONCATENACIÓN GLOBAL (OUTER JOIN) Y EXPORTACIÓN A PARQUET
==============================================================================
"""
# ==============================================================================
# PASO 3: CONCATENACIÓN OUTER EN MEMORIA
# ==============================================================================
print("\n--- PASO 3: Concatenando las 14 cohortes filtradas en memoria ---")
# Concatenación general con alineación externa de columnas
base_unida = pd.concat(lista_dfs, axis=0, ignore_index=True, join="outer")
# Liberar memoria RAM ocupada por la lista temporal
del lista_dfs
gc.collect()
print("=" * 75)
print("📊 RESULTADO DE LA CONCATENACIÓN REGIONAL MAESTRA")
print(f"   - Total Registros (N) - Bogotá y Cundinamarca: {base_unida.shape[0]:,} filas")
print(f"   - Total Columnas  (M) - Esquema Unificado:      {base_unida.shape[1]} columnas")
print("=" * 75)
# =============================================================================
# PASO 4: ALMACENAMIENTO PERSISTENTE EN GOOGLE DRIVE
# ==============================================================================
OUTPUT_DIR = Path(
    r"/content/drive/MyDrive/Maestria Ingenieria y analitica de datos/semestre 4/proyecto de grado/fuentes de datos/base_finales/base_parquet_1"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "icfes_saber11_consolidado_bogota_cundinamarca_2019_2025.parquet"

print("\n--- PASO 4: Exportación del dataset consolidado a Parquet (Snappy) ---")
base_unida.to_parquet(OUTPUT_FILE, engine="pyarrow", index=False)

print(f"✅ ¡Éxito! Base unida guardada exitosamente en:\n   -> {OUTPUT_FILE}")
print("=" * 75)


--- PASO 3: Concatenando las 14 cohortes filtradas en memoria ---
📊 RESULTADO DE LA CONCATENACIÓN REGIONAL MAESTRA
   - Total Registros (N) - Bogotá y Cundinamarca: 861,251 filas
   - Total Columnas  (M) - Esquema Unificado:      92 columnas

--- PASO 4: Exportación del dataset consolidado a Parquet (Snappy) ---
✅ ¡Éxito! Base unida guardada exitosamente en:
   -> /content/drive/MyDrive/Maestria Ingenieria y analitica de datos/semestre 4/proyecto de grado/fuentes de datos/base_finales/base_parquet_1/icfes_saber11_consolidado_bogota_cundinamarca_2019_2025.parquet


In [ ]:
"""
==============================================================================
PASO 5: AUDITORÍA LONGITUDINAL DE DATOS FALTANTES (NULOS) Y POBLAMIENTO
==============================================================================
"""

import pandas as pd

# 1. Calcular métricas por columna
total_filas = len(base_unida)
datos_validos = base_unida.count()
datos_faltantes = base_unida.isnull().sum()

porcentaje_faltantes = (datos_faltantes / total_filas) * 100
porcentaje_poblado = (datos_validos / total_filas) * 100

# 2. Métrica global de poblamiento de toda la base
total_celdas = base_unida.size
total_celdas_validas = datos_validos.sum()
pct_poblamiento_global = (total_celdas_validas / total_celdas) * 100

# 3. Consolidar reporte en un DataFrame
reporte_nulos = pd.DataFrame(
    {
        "Datos Válidos": datos_validos,
        "Datos Faltantes": datos_faltantes,
        "% Faltantes": porcentaje_faltantes.map("{:.2f}%".format),
        "% Poblado": porcentaje_poblado.map("{:.2f}%".format),
    }
).sort_values(by="Datos Faltantes", ascending=False)

# 4. Visualizar el reporte completo
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 1000)

print("=" * 80)
print(
    f"📊 AUDITORÍA GENERAL DE NULOS Y POBLAMIENTO "
    f"(Total Filas: {total_filas:,} | Columnas: {base_unida.shape[1]})"
)
print(f"📈 Tasa de Población Global del Dataset: {pct_poblamiento_global:.2f}%")
print("=" * 80)
print(reporte_nulos)
print("=" * 80)

# Restablecer visualización por defecto
pd.reset_option("display.max_rows")
pd.reset_option("display.width")

📊 AUDITORÍA GENERAL DE NULOS Y POBLAMIENTO (Total Filas: 861,251 | Columnas: 92)
📈 Tasa de Población Global del Dataset: 90.21%
                               Datos Válidos  Datos Faltantes % Faltantes % Poblado
fami_posicionhermanos                  88103           773148      89.77%    10.23%
estu_horastrabnoremu                  115503           745748      86.59%    13.41%
estu_desplazacolegio                  117782           743469      86.32%    13.68%
estu_tiempocasaacole                  117782           743469      86.32%    13.68%
estu_numhijos                         117799           743452      86.32%    13.68%
fami_numhermanos                      117799           743452      86.32%    13.68%
estu_comunidadcampesina               117799           743452      86.32%    13.68%
estu_grupoetnia                       118852           742399      86.20%    13.80%
estu_generacione                      362274           498977      57.94%    42.06%
cole_bilingue                   

In [ ]:
"""
==============================================================================
PASO 6: DEPURACIÓN DE VARIABLES (MENOS DEL 80% DE COMPLETITUD / >20% NULOS)
==============================================================================
"""

import pandas as pd

# 1. Definir umbral de completitud mínima exigida (80% poblado = máx 20% nulos)
MIN_COMPLETITUD = 80.0
MAX_NULOS_PERMITIDOS = 100.0 - MIN_COMPLETITUD  # 20.0%

total_filas = len(base_unida)
pct_faltantes = (base_unida.isnull().sum() / total_filas) * 100
pct_poblado = (base_unida.count() / total_filas) * 100

# 2. Identificar las columnas a eliminar
cols_a_eliminar = pct_faltantes[
    pct_faltantes > MAX_NULOS_PERMITIDOS
].index.tolist()

print("=" * 80)
print(
    f"📊 DEPURACIÓN: Eliminando variables con completitud menor al {MIN_COMPLETITUD}%"
)
print(f"   (Equivale a más del {MAX_NULOS_PERMITIDOS}% de valores nulos)")
print("=" * 80)

print(f"⚠️ Se eliminarán {len(cols_a_eliminar)} columnas:")
for col in cols_a_eliminar:
    print(
        f"   ❌ {col:<32} | {pct_poblado[col]:.2f}% poblado ({pct_faltantes[col]:.2f}% nulos)"
    )

# 3. Eliminar columnas del DataFrame
base_unida.drop(columns=cols_a_eliminar, inplace=True)

print("-" * 80)
print("✅ Depuración completada exitosamente.")
print(
    f"📐 Dimensiones actuales de base_unida: {base_unida.shape[0]:,} filas × {base_unida.shape[1]} columnas."
)
print("=" * 80)

# 4. Verificación inmediata post-depuración (Auditoría de las columnas que quedaron)
datos_validos_post = base_unida.count()
datos_faltantes_post = base_unida.isnull().sum()
pct_faltantes_post = (datos_faltantes_post / total_filas) * 100
pct_poblado_post = (datos_validos_post / total_filas) * 100

reporte_post = pd.DataFrame(
    {
        "Datos Válidos": datos_validos_post,
        "Datos Faltantes": datos_faltantes_post,
        "% Faltantes": pct_faltantes_post.map("{:.2f}%".format),
        "% Poblado": pct_poblado_post.map("{:.2f}%".format),
    }
).sort_values(by="Datos Faltantes", ascending=False)

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 1000)

print("\n" + "=" * 80)
print("📋 REPORTE DE COLUMNAS RESTANTES (Todas con >= 80% de poblamiento)")
print("=" * 80)
print(reporte_post)
print("=" * 80)

pd.reset_option("display.max_rows")
pd.reset_option("display.width")

📊 DEPURACIÓN: Eliminando variables con completitud menor al 80.0%
   (Equivale a más del 20.0% de valores nulos)
⚠️ Se eliminarán 10 columnas:
   ❌ cole_bilingue                    | 79.29% poblado (20.71% nulos)
   ❌ estu_grupoetnia                  | 13.80% poblado (86.20% nulos)
   ❌ estu_generacione                 | 42.06% poblado (57.94% nulos)
   ❌ fami_numhermanos                 | 13.68% poblado (86.32% nulos)
   ❌ estu_comunidadcampesina          | 13.68% poblado (86.32% nulos)
   ❌ estu_numhijos                    | 13.68% poblado (86.32% nulos)
   ❌ estu_horastrabnoremu             | 13.41% poblado (86.59% nulos)
   ❌ fami_posicionhermanos            | 10.23% poblado (89.77% nulos)
   ❌ estu_tiempocasaacole             | 13.68% poblado (86.32% nulos)
   ❌ estu_desplazacolegio             | 13.68% poblado (86.32% nulos)
--------------------------------------------------------------------------------
✅ Depuración completada exitosamente.
📐 Dimensiones actuales de base_unida: 

In [ ]:
print("=" * 70)
print(f"📐 DIMENSIONES ACTUALES DE base_unida:")
print(f"   - Registros (filas):    {base_unida.shape[0]:,}")
print(f"   - Atributos (columnas): {base_unida.shape[1]}")
print("=" * 70)

print("\n📋 LISTA DE LAS 84 COLUMNAS CONSERVADAS:")
for i, col in enumerate(base_unida.columns, 1):
    print(f" {i:2d}. {col}")

📐 DIMENSIONES ACTUALES DE base_unida:
   - Registros (filas):    861,251
   - Atributos (columnas): 82

📋 LISTA DE LAS 84 COLUMNAS CONSERVADAS:
  1. periodo
  2. estu_consecutivo
  3. estu_estudiante
  4. estu_tipodocumento
  5. cole_area_ubicacion
  6. cole_calendario
  7. cole_caracter
  8. cole_cod_dane_establecimiento
  9. cole_cod_dane_sede
 10. cole_cod_depto_ubicacion
 11. cole_cod_mcpio_ubicacion
 12. cole_codigo_icfes
 13. cole_depto_ubicacion
 14. cole_genero
 15. cole_jornada
 16. cole_mcpio_ubicacion
 17. cole_naturaleza
 18. cole_nombre_establecimiento
 19. cole_nombre_sede
 20. cole_sede_principal
 21. desemp_c_naturales
 22. desemp_ingles
 23. desemp_lectura_critica
 24. desemp_matematicas
 25. desemp_sociales_ciudadanas
 26. estu_agregado
 27. estu_cod_depto_presentacion
 28. estu_cod_mcpio_presentacion
 29. estu_cod_reside_depto
 30. estu_cod_reside_mcpio
 31. estu_dedicacioninternet
 32. estu_dedicacionlecturadiaria
 33. estu_depto_presentacion
 34. estu_depto_reside


# base sin faltantes(85%)

In [ ]:
"""
==============================================================================
EXPORTACIÓN DE LA BASE DEPURADA A GOOGLE DRIVE (PARQUET)
Ruta: .../fuentes de datos/base_finales/
==============================================================================
"""

from pathlib import Path

# 1. Definir la ruta de destino exacta
OUTPUT_DIR = Path(
    r"/content/drive/MyDrive/Maestria Ingenieria y analitica de datos/semestre 4/proyecto de grado/fuentes de datos/base_finales/base_sin_nulos/"
)

# Crear la carpeta si no existe
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Nombre del archivo Parquet final
OUTPUT_FILE = (
    OUTPUT_DIR / "icfes_saber11_bogota_cundinamarca_depurado_2019_2025.parquet"
)

print("⏳ Exportando base_unida (275,058 filas x 84 columnas) a Parquet...")

# 3. Guardar el DataFrame a Parquet con compresión Snappy
base_unida.to_parquet(OUTPUT_FILE, engine="pyarrow", index=False)

print("=" * 75)
print("💾 ¡ÉXITO! ARCHIVO DEPURADO EXPORTADO EN DRIVE")
print(f"   -> Ruta: {OUTPUT_FILE}")
print("=" * 75)

⏳ Exportando base_unida (275,058 filas x 84 columnas) a Parquet...
💾 ¡ÉXITO! ARCHIVO DEPURADO EXPORTADO EN DRIVE
   -> Ruta: /content/drive/MyDrive/Maestria Ingenieria y analitica de datos/semestre 4/proyecto de grado/fuentes de datos/base_finales/base_sin_nulos/icfes_saber11_bogota_cundinamarca_depurado_2019_2025.parquet


In [ ]:
base_unida.shape

(861251, 82)

# seleccion variables finales

In [ ]:
"""
==============================================================================
PASO 7: FILTRADO DE VARIABLES
==============================================================================
"""

import pandas as pd

# ----------------------------------------------------------------------------
# 1. RENOMBRAR Y LIBERAR MEMORIA
# ----------------------------------------------------------------------------
base_filtrada = base_unida.copy()

# Liberar la variable previa para optimizar memoria RAM
del base_unida

# ----------------------------------------------------------------------------
# 2. DEFINIR LISTA DE COLUMNAS A ELIMINAR
# ----------------------------------------------------------------------------
cols_a_eliminar = [
    # Identificadores (no geográficos)
    "estu_estudiante",
    "estu_agregado",
    "cole_cod_dane_establecimiento",
    "cole_cod_dane_sede",
    "estu_cod_depto_presentacion",

    # Niveles de desempeño
    "desemp_c_naturales",
    "desemp_ingles",
    "desemp_lectura_critica",
    "desemp_matematicas",
    "desemp_sociales_ciudadanas",
    # Percentiles
    "percentil_c_naturales",
    "percentil_global",
    "percentil_ingles",
    "percentil_lectura_critica",
    "percentil_matematicas",
    "percentil_sociales_ciudadanas",
]

# ----------------------------------------------------------------------------
# 3. FILTRAR COLUMNAS Y CONSERVAR EN VARIABLE
# ----------------------------------------------------------------------------
cols_existentes = [col for col in cols_a_eliminar if col in base_filtrada.columns]
base_filtrada.drop(columns=cols_existentes, inplace=True)

print("=" * 80)
print("📊 BASE RESTRUCTURADA Y FILTRADA CON ÉXITO")
print(f"   - Variable asignada:   base_filtrada")
print(f"   - Dimensiones:         {base_filtrada.shape[0]:,} filas × {base_filtrada.shape[1]} columnas")
print("=" * 80)

📊 BASE RESTRUCTURADA Y FILTRADA CON ÉXITO
   - Variable asignada:   base_filtrada
   - Dimensiones:         861,251 filas × 66 columnas


In [ ]:
base_filtrada.shape

(861251, 66)

In [ ]:
"""
==============================================================================
PASO 8: CREACIÓN DE CARPETA 'base_filtrada' Y GUARDADO DEL PARQUET
==============================================================================
"""

from pathlib import Path
import pandas as pd

# 1. Asignar/Asegurar la variable 'base_filtrada' en el notebook
if "base_unida" in globals():
    base_filtrada = base_unida.copy()
    del base_unida  # Liberar memoria de la variable anterior

# 2. Definir la nueva carpeta dentro de 'base_finales'
NUEVA_CARPETA = Path(
    r"/content/drive/MyDrive/Maestria Ingenieria y analitica de datos/semestre 4/proyecto de grado/fuentes de datos/base_finales/base_filtrada"
)

# Crear la carpeta física en Google Drive si no existe
NUEVA_CARPETA.mkdir(parents=True, exist_ok=True)

# 3. Definir la ruta del archivo Parquet
ARCHIVO_PARQUET = NUEVA_CARPETA / "base_filtrada.parquet"

# 4. Guardar la base de datos
print("⏳ Guardando 'base_filtrada' en Google Drive...")
base_filtrada.to_parquet(ARCHIVO_PARQUET, engine="pyarrow", index=False)

print("=" * 80)
print("📁 ¡PROCESO COMPLETADO EXITOSAMENTE!")
print(f"   - Carpeta creada: .../base_finales/base_filtrada/")
print(f"   - Archivo guardado: {ARCHIVO_PARQUET.name}")
print(f"   - Dimensiones: {base_filtrada.shape[0]:,} filas × {base_filtrada.shape[1]} columnas")
print("=" * 80)

⏳ Guardando 'base_filtrada' en Google Drive...
📁 ¡PROCESO COMPLETADO EXITOSAMENTE!
   - Carpeta creada: .../base_finales/base_filtrada/
   - Archivo guardado: base_filtrada.parquet
   - Dimensiones: 861,251 filas × 66 columnas
